**Experiment 2 (notes_2D.pdf): learning the 2D energy functional with the DM21-like model.**

Identical to the LERN setup (two reweighted energy terms: bare kinetic + unscreened Coulomb), except
the feature vector additionally contains the nonlocal unscreened Hartree energy density. Expected:
succeeds also for $\beta \neq 0$, $q_s = 0$; degrades for $q_s > 0$.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
import math, os, time, copy
import pandas as pd
from mpl_toolkits.axes_grid1 import make_axes_locatable
from ewaldnn2d import *

torch.random.manual_seed(1234) # for reproducibility

# Global settings
dtype = torch.float64
device = "cpu"
N_batch = 100
N_epochs = 10000
lr = 1e-1 # we will use a LR scheduler, so this is just an initial value
min_delta = 1e-5 # min change in the monitored quantity to qualify as an improvement
patience = 100    # epochs to wait for improvement before stopping training
pad_mode = "reflect" # in this example, we use reflective padding for local feature generation
N_pow = 1 # number of local features per grid point
N_train = 1500
N_test = 250
N_val = 250
N_energy_terms = 2 # number of energy terms: bare kinetic + unscreened Coulomb

# grid and basis settings
N_x = 32 # number of grid points in x direction
N_y = N_x # number of grid points in y direction
m_x = torch.arange(0, N_x, dtype=dtype, device=device)             # (N_x,)
m_y = torch.arange(0, N_y, dtype=dtype, device=device)             # (N_y,)
abs_val = torch.sqrt(m_x[:, None]**2 + m_y[None, :]**2)  # (M_x, M_y)
x = torch.linspace(0, 1, N_x, dtype=dtype, device=device)            # (N_x,)
y = torch.linspace(0, 1, N_y, dtype=dtype, device=device)            # (N_y,)
DM_x = torch.cos(torch.pi * torch.outer(m_x, x))                  # (M_x, N_x)
DM_y = torch.cos(torch.pi * torch.outer(m_y, y))                  # (M_y, N_y)
DerDM_x = -torch.pi * m_x[:, None] * torch.sin(torch.pi * torch.outer(m_x, x))  # (M_x, N_x) # derivative of design matrix
DerDM_y = -torch.pi * m_y[:, None] * torch.sin(torch.pi * torch.outer(m_y, y))  # (M_y, N_y) # derivative of design matrix

data_regime = "smooth" # "smooth" or "rough"
if data_regime == "smooth":
    M_cutoff = 10 # maximum harmonic
    std_harm = 2.0 / (1.0 + 0.2 * abs_val)**2 * (abs_val <= M_cutoff).double()  # (M_x, M_y)
elif data_regime == "rough":
    std_harm = 2.0 / (1.0 + 0.0 * abs_val)**2 # (M_x, M_y)
else:
    raise ValueError("regime must be 'smooth' or 'rough'")
std_harm[0, 0] = 0.0 # no uniform density offset

# local stiffness parameters; chosen so that kappa spreads over (0, 2)
# without saturating the tanh (see the diagnostic histogram below). Scales of the
# tanh argument terms (smooth regime, 32x32): |4-neighbor product| ~ 8 (median),
# phi std ~ 975 / 515 / 50 for qs = 0 / 0.1 / 0.5
alpha = 0.005
beta = -0.002

# interaction kernel parameters
qs = 0.1 # screening momentum
amp = 1.0 # amplitude of interaction kernel
kernel_regime = "screened_coulomb"

# bare (kappa = 1) local kinetic energy density - fed to the models
def E_kin_loc(rho, d_rho_x, d_rho_y, eng_dens_flag=False):
    return E_kin_custom(rho, d_rho_x, d_rho_y, alpha=0.0, beta=0.0, qs=qs, eng_dens_flag=eng_dens_flag)

# unscreened Coulomb (Hartree) local energy density - fed to the models
def E_HF_loc(rho, d_rho_x, d_rho_y, eng_dens_flag=False):
    return amp * E_int_ms_dct(rho, kernel=kernel_regime, eng_dens_flag=eng_dens_flag, qs=0.0)

# target: kinetic term with stiffness kappa[alpha, beta] + screened interaction
def E_tot(rho, d_rho_x, d_rho_y):
    return E_kin_custom(rho, d_rho_x, d_rho_y, alpha=alpha, beta=beta, qs=qs) \
         + amp * E_int_ms_dct(rho, kernel=kernel_regime, qs=qs)

# generate train/test split (all three Experiment 2 notebooks share this dataset)
fname = f"DATA2d/EngFunc_dataset_{data_regime}_{kernel_regime}_{alpha}_{beta}_{qs}_{amp}_{N_x}_{N_y}.pt"
flag_generate_data = not os.path.isfile(fname) # generate once; set to True to force regeneration
if flag_generate_data:
    N_batch_int = 10 # number of density profiles per data generation batch
    torch.manual_seed(1234) # for reproducibility
    rho_train, d_rho_x_train, d_rho_y_train, a_train, E_loc_kin_train, E_loc_HF_train, E_tot_train = generate_EngFunc_data_2d(N_train, N_batch_int, E_kin_loc, E_HF_loc, E_tot, std_harm=std_harm, DM_x=DM_x, DerDM_x=DerDM_x, DM_y=DM_y, DerDM_y=DerDM_y)
    rho_test, d_rho_x_test, d_rho_y_test, a_test, E_loc_kin_test, E_loc_HF_test, E_tot_test = generate_EngFunc_data_2d(N_test, N_batch_int, E_kin_loc, E_HF_loc, E_tot, std_harm=std_harm, DM_x=DM_x, DerDM_x=DerDM_x, DM_y=DM_y, DerDM_y=DerDM_y)
    rho_val, d_rho_x_val, d_rho_y_val, a_val, E_loc_kin_val, E_loc_HF_val, E_tot_val = generate_EngFunc_data_2d(N_val, N_batch_int, E_kin_loc, E_HF_loc, E_tot, std_harm=std_harm, DM_x=DM_x, DerDM_x=DerDM_x, DM_y=DM_y, DerDM_y=DerDM_y)
    # save data to disk
    os.makedirs("DATA2d", exist_ok=True)

    torch.save(
        {
            "rho_train": rho_train,
            "d_rho_x_train": d_rho_x_train,
            "d_rho_y_train": d_rho_y_train,
            "a_train": a_train,
            "E_loc_kin_train": E_loc_kin_train,
            "E_loc_HF_train": E_loc_HF_train,
            "E_tot_train": E_tot_train,
            "rho_val": rho_val,
            "d_rho_x_val": d_rho_x_val,
            "d_rho_y_val": d_rho_y_val,
            "a_val": a_val,
            "E_loc_kin_val": E_loc_kin_val,
            "E_loc_HF_val": E_loc_HF_val,
            "E_tot_val": E_tot_val,
            "rho_test": rho_test,
            "d_rho_x_test": d_rho_x_test,
            "d_rho_y_test": d_rho_y_test,
            "a_test": a_test,
            "E_loc_kin_test": E_loc_kin_test,
            "E_loc_HF_test": E_loc_HF_test,
            "E_tot_test": E_tot_test,
            "data_regime": data_regime,
            "kernel_regime": kernel_regime,
            "alpha": alpha,
            "beta": beta,
            "qs": qs,
            "amp": amp,
        },
        fname,
    )
else:
    data = torch.load(fname)
    rho_train = data["rho_train"]
    E_loc_kin_train = data["E_loc_kin_train"]
    E_loc_HF_train = data["E_loc_HF_train"]
    E_tot_train = data["E_tot_train"]
    rho_test = data["rho_test"]
    E_loc_kin_test = data["E_loc_kin_test"]
    E_loc_HF_test = data["E_loc_HF_test"]
    E_tot_test = data["E_tot_test"]
    rho_val = data["rho_val"]
    E_loc_kin_val = data["E_loc_kin_val"]
    E_loc_HF_val = data["E_loc_HF_val"]
    E_tot_val = data["E_tot_val"]

features_train = generate_loc_features_rs(rho_train, N_pow=N_pow)  # (N_train, N_x, N_y, N_pow)
features_test  = generate_loc_features_rs(rho_test, N_pow=N_pow)   # (N_test, N_x, N_y, N_pow)
features_val   = generate_loc_features_rs(rho_val, N_pow=N_pow)    # (N_val, N_x, N_y, N_pow)

# Extend features with neighbor information
R_feat = 1.0 # radius for neighbor feature extension, 1.0 or 1.5
features_train = extend_features_neighbors_2d(features_train, R=R_feat)
features_test  = extend_features_neighbors_2d(features_test, R=R_feat)
features_val   = extend_features_neighbors_2d(features_val, R=R_feat)

# This element is new: we append the local HF energy which will enter as a feature into the DM21 model
# Note that we append the unnormalized HF energy, and then normalize the entire feature set together (including the HF energy term)
features_train = torch.cat([features_train, E_loc_HF_train], dim=-1)  # (N_train, N_x, N_y, N_feat + 1)
features_test  = torch.cat([features_test,  E_loc_HF_test], dim=-1)   # (N_test,  N_x, N_y, N_feat + 1)
features_val   = torch.cat([features_val,   E_loc_HF_val], dim=-1)    # (N_val,   N_x, N_y, N_feat + 1)

# Normalize features
mean_feat, std_feat = compute_normalization_stats(features_train)
features_train_norm = normalize_features(features_train, mean_feat, std_feat)
features_test_norm = normalize_features(features_test, mean_feat, std_feat)
features_val_norm = normalize_features(features_val, mean_feat, std_feat)

# Supplement the features with the two unnormalized energy terms (bare kinetic + unscreened Coulomb)
features_train_norm = torch.cat([features_train_norm, E_loc_kin_train, E_loc_HF_train], dim=-1)  # (N_train, N_x, N_y, N_feat + 2)
features_test_norm  = torch.cat([features_test_norm,  E_loc_kin_test,  E_loc_HF_test], dim=-1)   # (N_test,  N_x, N_y, N_feat + 2)
features_val_norm   = torch.cat([features_val_norm,   E_loc_kin_val,   E_loc_HF_val], dim=-1)    # (N_val,   N_x, N_y, N_feat + 2)

# Normalize targets
E_mean = E_tot_train.mean()
E_std = E_tot_train.std()
E_tot_train_norm = (E_tot_train - E_mean) / E_std
E_tot_test_norm = (E_tot_test - E_mean) / E_std
E_tot_val_norm = (E_tot_val - E_mean) / E_std
# Datasets
train_dataset = TensorDataset(features_train_norm, E_tot_train_norm)
val_dataset   = TensorDataset(features_val_norm,   E_tot_val_norm)
test_dataset  = TensorDataset(features_test_norm,  E_tot_test_norm)
# Loaders
train_loader = DataLoader(train_dataset, batch_size=N_batch, shuffle=True,  drop_last=False)
val_loader   = DataLoader(val_dataset,   batch_size=N_batch, shuffle=False, drop_last=False)
test_loader  = DataLoader(test_dataset,  batch_size=N_batch, shuffle=False, drop_last=False)

_, _, _, N_feat = features_train.shape
print(f"Number of local features per grid point: {N_feat}")

In [ ]:
# visualize a sampled density profile and the true local stiffness kappa_r
k = 0  # index of the test sample to visualize
kappa_test = kappa_custom(rho_test, alpha=alpha, beta=beta, qs=qs)  # (N_test, N_x, N_y)

fig, axes = plt.subplots(1, 2, figsize=(9, 4), constrained_layout=True)
im0 = axes[0].imshow(
    rho_test[k].numpy().T,              # transpose so x is horizontal, y vertical
    origin="lower",
    extent=[0, 1, 0, 1],
    aspect="equal"
)
axes[0].set_title(rf"$\rho(x,y)$ ({data_regime})")
axes[0].set_xlabel(r"$x$")
axes[0].set_ylabel(r"$y$")
fig.colorbar(im0, ax=axes[0], fraction=0.046)
im1 = axes[1].imshow(
    kappa_test[k].numpy().T,
    origin="lower",
    extent=[0, 1, 0, 1],
    aspect="equal",
    vmin=0.0, vmax=2.0
)
axes[1].set_title(rf"true $\varkappa(x,y)$, $\alpha$={alpha}, $\beta$={beta}, $q_s$={qs}")
axes[1].set_xlabel(r"$x$")
axes[1].set_ylabel(r"$y$")
fig.colorbar(im1, ax=axes[1], fraction=0.046)
plt.show()

# pooled distribution of the true stiffness - a diagnostic for choosing alpha and beta:
# kappa should neither saturate at {0, 2} nor collapse onto 1
plt.figure(figsize=(5, 4))
plt.hist(kappa_test.reshape(-1).numpy(), bins=np.linspace(0.0, 2.0, 101), density=True, alpha=0.8)
plt.xlabel(r"true stiffness $\varkappa(x,y)$")
plt.ylabel("probability density")
plt.title(f"Distribution of the true local stiffness\n"
          rf"(pooled over {N_test} samples and space)")
plt.tight_layout()
plt.show()

In [ ]:
ckpt_dir = "LearningEngFunc2d_checkpoints"
flag_train = True  # set to True to train models
learning_regime = "LERN2d"

n_hidden_list = [1, 2, 3, 4]
n_neurons_list = [8, 16, 32, 64]

# NB: the hyperparameter grid also spans N_feat in [6, 10]; to cover it, run the
# notebook twice: once with R_feat = 1.0 (6 features) and once with R_feat = 1.5
# (10 features) in the cell above.

if flag_train:
    for n_hidden in n_hidden_list:
        for n_neurons in n_neurons_list:
            for seed in range(3): # train a few models with different random seeds

                torch.manual_seed(1234 + seed)
                run_name = f"LERN2d_mod_EngFunc_" + data_regime + '_' + kernel_regime + f"_{alpha}_{beta}_{qs}_{amp}_{N_x}_{N_y}_{N_feat}_{N_energy_terms}_{n_hidden}_{n_neurons}_{seed}"
                print(f"Training model: n_hidden={n_hidden}, n_neurons={n_neurons}, seed={seed}")

                model = LERN2d(
                        N_x=N_x,
                        N_y=N_y,
                        N_energy_terms=N_energy_terms,
                        N_feat=N_feat,
                        n_hidden=n_hidden,
                        n_neurons=n_neurons,
                        mean_feat=mean_feat,
                        std_feat=std_feat,
                        E_mean=E_mean,
                        E_std=E_std,
                    ).to(device=device, dtype=dtype)

                optimizer = optim.Adam(model.parameters(), lr=lr)
                criterion = nn.MSELoss()

                # Reduce LR when val loss plateaus
                scheduler = optim.lr_scheduler.ReduceLROnPlateau(
                    optimizer, mode='min', factor=0.5, patience=50, cooldown=2, min_lr=1e-6
                )

                hist, best_epoch = train_with_early_stopping(
                    model=model,
                    train_loader=train_loader,
                    val_loader=val_loader,
                    criterion=criterion,
                    optimizer=optimizer,
                    scheduler=scheduler,
                    max_epochs=N_epochs,
                    patience=patience,
                    min_delta=min_delta,
                    ckpt_dir=ckpt_dir,
                    run_name=run_name,
                    learning_regime=learning_regime,
                    N_x=N_x,
                    N_y=N_y,
                    device=device,
                )

In [ ]:
best_val = math.inf
n_hidden_best = 0
n_neurons_best = 0
seed_best = 0
N_feat_best = 0

N_feat_list = [6, 10]
for n_hidden in n_hidden_list:
    for n_neurons in n_neurons_list:
        for seed in range(3): # trained 3 models with different random seeds
            for N_feat in N_feat_list:

                run_name = f"LERN2d_mod_EngFunc_" + data_regime + '_' + kernel_regime + f"_{alpha}_{beta}_{qs}_{amp}_{N_x}_{N_y}_{N_feat}_{N_energy_terms}_{n_hidden}_{n_neurons}_{seed}"
                model, normalization, epoch, val_loss = load_checkpoint(
                    ckpt_dir + f"/{run_name}_best.pt",
                    LERN2d,
                    device=device
                    )
                model = model.to(device=device, dtype=dtype)

                if val_loss < best_val:
                    best_val = val_loss
                    n_hidden_best = n_hidden
                    n_neurons_best = n_neurons
                    seed_best = seed
                    N_feat_best = N_feat

print("\n")
print("Overall best model:")
print(f"Best model: n_hidden={n_hidden_best}, n_neurons={n_neurons_best}, seed={seed_best}, N_feat={N_feat_best}, val_loss={best_val:.10f}")

In [ ]:
n_hidden = n_hidden_best
n_neurons = n_neurons_best
seed = seed_best
N_feat = N_feat_best

run_name = f"LERN2d_mod_EngFunc_" + data_regime + '_' + kernel_regime + f"_{alpha}_{beta}_{qs}_{amp}_{N_x}_{N_y}_{N_feat}_{N_energy_terms}_{n_hidden}_{n_neurons}_{seed}"

path = ckpt_dir + f"/{run_name}_history.csv"
hist_df = pd.read_csv(path)
print(hist_df.head())
hist_df.plot(x="epoch", y=["train_loss", "val_loss"], logy=True, grid=True, title=run_name)

os.makedirs("Results_2D/Experiment_2", exist_ok=True)
data = np.column_stack([
    hist_df["epoch"].to_numpy(),
    hist_df["train_loss"].to_numpy(),
    hist_df["val_loss"].to_numpy(),
])
np.savetxt(f"Results_2D/Experiment_2/DM21_performance_{data_regime}_{alpha}_{beta}_{qs}.txt", data)
print(f"Best model: n_hidden={n_hidden_best}, n_neurons={n_neurons_best}, seed={seed_best}, N_feat={N_feat_best}")
print(f"Best validation loss: {min(data[:, 2])}")

In [ ]:
model, normalization, epoch, val_loss = load_checkpoint(
        ckpt_dir + f"/{run_name}_best.pt",
        LERN2d,
        device=device
    )
model = model.to(device=device, dtype=dtype)

# Rebuild the test features to match the best model's feature set:
# N_feat = 6 -> R_feat = 1.0 (4 nearest neighbors), N_feat = 10 -> R_feat = 1.5 (8 neighbors);
# the nonlocal HF energy is part of the DM21 feature vector, appended before normalization.
# Normalization stats come from the checkpoint, as they were computed for that feature set.
R_feat_best = 1.0 if N_feat == 6 else 1.5
features_test_best = extend_features_neighbors_2d(generate_loc_features_rs(rho_test, N_pow=N_pow), R=R_feat_best)
features_test_best = torch.cat([features_test_best, E_loc_HF_test], dim=-1)
features_test_best_norm = normalize_features(features_test_best, normalization["mean_feat"], normalization["std_feat"])

k = 0  # index of the test sample to visualize
factors = model.local_nn(features_test_best_norm[k:k+1])  # (1, N_x, N_y, 2): [f_kin, f_int]

# if learning succeeded, f_kin should track the true stiffness kappa;
# for qs = 0 the exact solution is f_kin = kappa and f_int = 1
fig, axes = plt.subplots(1, 3, figsize=(13, 4), constrained_layout=True)
maps = [
    (kappa_test[k].numpy().T, r"true $\varkappa(x,y)$"),
    (factors[0, :, :, 0].detach().numpy().T, r"$f_{\rm kin}(x,y)$"),
    (factors[0, :, :, 1].detach().numpy().T, r"$f_{\rm int}(x,y)$"),
]
for ax, (arr, title) in zip(axes, maps):
    im = ax.imshow(arr, origin="lower", extent=[0, 1, 0, 1], aspect="equal", vmin=0.0, vmax=2.0)
    ax.set_title(title)
    ax.set_xlabel(r"$x$")
    ax.set_ylabel(r"$y$")
    fig.colorbar(im, ax=ax, fraction=0.046)
plt.show()

In [ ]:
f_kin_all, f_int_all = [], []
for k in range(N_test):
    factors = model.local_nn(features_test_best_norm[k:k+1])  # (1, N_x, N_y, 2)
    f_kin_all.append(factors[..., 0].reshape(-1).detach().numpy())
    f_int_all.append(factors[..., 1].reshape(-1).detach().numpy())
f_kin_all = np.concatenate(f_kin_all)
f_int_all = np.concatenate(f_int_all)

bins = np.linspace(0.0, 2.0, 101)
plt.figure(figsize=(5, 4))
plt.hist(kappa_test.reshape(-1).numpy(), bins=bins, density=True, alpha=0.5, label=r"true $\varkappa$")
plt.hist(f_kin_all, bins=bins, density=True, alpha=0.5, label=r"$f_{\rm kin}$")
plt.hist(f_int_all, bins=bins, density=True, alpha=0.5, label=r"$f_{\rm int}$")
plt.xlim(-0.1, 2.1)
plt.xlabel("local factor value")
plt.ylabel("probability density")
plt.legend()
plt.title(f"Learned local factors vs true stiffness\n"
          rf"(pooled over {N_test} samples and space)")
plt.tight_layout()
plt.show()